# Combined-feature selection experiment

This notebook compares the full combined feature set against Mutual Information, RFECV, and Boruta selection for network-intrusion classification. Mutual Information and RFECV are evaluated with Logistic Regression, Decision Tree, Random Forest, XGBoost, and NGBoost. Boruta is evaluated only with Random Forest and XGBoost.

MLflow run names follow `Combined_{FeatureSelection}_{Algorithm}`, for example `Combined_RFECV_LogisticReg`. Selection is learned only from training data; the held-out test split is never used to select features.

In [ ]:
import json
import platform
import sys
import time
from collections import Counter
from functools import partial
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import ngboost
import numpy as np
import pandas as pd
import sklearn
import xgboost
from boruta import BorutaPy
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from ngboost import NGBClassifier
from sklearn.base import ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE

TRACKING_DB = (PROJECT_ROOT / "Notebooks" / "mlflow.db").resolve()
TRACKING_URI = f"sqlite:///{TRACKING_DB.as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

SELECTION_SAMPLE_SIZE = 30_000
MI_TOP_K = 36
RFECV_FOLDS = 3
RFECV_STEP = 0.10
RFECV_MIN_FEATURES = 10
BORUTA_MAX_ITER = 25
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
data_path = PROJECT_ROOT / "Data" / "Consolidated_df.csv"
df = pd.read_csv(data_path)

ORIGINAL_FEATURES = (
    "duration", "protocoltype", "service", "flag", "srcbytes",
    "dstbytes", "land", "wrongfragment", "urgent", "hot",
    "numfailedlogins", "loggedin", "numcompromised", "rootshell",
    "suattempted", "numroot", "numfilecreations", "numshells",
    "numaccessfiles", "numoutboundcmds", "ishostlogin",
    "isguestlogin", "count", "srvcount", "serrorrate",
    "srvserrorrate", "rerrorrate", "srvrerrorrate", "samesrvrate",
    "diffsrvrate", "srvdiffhostrate", "dsthostcount",
    "dsthostsrvcount", "dsthostsamesrvrate", "dsthostdiffsrvrate",
    "dsthostsamesrcportrate", "dsthostsrvdiffhostrate",
    "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate",
)
ENGINEERED_FEATURES = (
    "total_bytes", "bytes_per_second", "src_dst_byte_ratio",
    "src_byte_fraction", "dst_byte_fraction", "byte_asymmetry",
    "service_connection_ratio", "host_service_ratio",
    "different_service_connections", "different_host_service_connections",
    "short_term_scan_pressure", "host_scan_pressure",
    "same_source_port_pressure", "short_term_serror_score",
    "short_term_rerror_score", "short_term_error_score",
    "host_serror_score", "host_rerror_score", "host_error_score",
    "same_service_rate_gap", "different_service_rate_gap",
    "serror_rate_gap", "rerror_rate_gap",
    "authentication_risk_score", "has_failed_login",
    "failed_login_and_logged_in", "suspicious_admin_activity",
    "privileged_activity_score", "content_risk_score",
    "file_and_shell_activity", "root_compromise_ratio",
)
COMBINED_FEATURES = ORIGINAL_FEATURES + ENGINEERED_FEATURES
assert len(COMBINED_FEATURES) == len(set(COMBINED_FEATURES))
assert set(COMBINED_FEATURES).issubset(df.columns)

X = df[list(COMBINED_FEATURES)]
y = df["binary_target"]
X_train_combined, X_test_combined, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
label_mapping = {"Normal": 0, "Attack": 1}
y_train_encoded = y_train.astype(str).map(label_mapping)
y_test_encoded = y_test.astype(str).map(label_mapping)
if y_train_encoded.isna().any() or y_test_encoded.isna().any():
    raise ValueError("Unexpected binary_target labels")

categorical_features = X_train_combined.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numeric_features = [c for c in COMBINED_FEATURES if c not in categorical_features]
print(
    f"Train={X_train_combined.shape}, test={X_test_combined.shape}, "
    f"original={len(ORIGINAL_FEATURES)}, engineered={len(ENGINEERED_FEATURES)}"
)

In [ ]:
def make_selection_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1
            )),
        ]), categorical_features),
    ], remainder="drop", verbose_feature_names_out=False)

def make_model_preprocessor(selected_features):
    selected_numeric = [c for c in selected_features if c in numeric_features]
    selected_categorical = [c for c in selected_features if c in categorical_features]
    transformers = []
    if selected_numeric:
        transformers.append(("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), selected_numeric))
    if selected_categorical:
        transformers.append(("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), selected_categorical))
    return ColumnTransformer(
        transformers, remainder="drop", verbose_feature_names_out=False
    )

class CompatibleNGBClassifier(ClassifierMixin, NGBClassifier):
    def fit(self, X, y, **fit_params):
        fitted = super().fit(X, y, **fit_params)
        self.classes_ = np.unique(y)
        return fitted

def make_models():
    return {
        "LogisticReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, objective="binary:logistic",
            eval_metric="logloss", tree_method="hist",
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "NGBoost": CompatibleNGBClassifier(
            n_estimators=200, learning_rate=0.05,
            random_state=RANDOM_STATE, verbose=False,
        ),
    }

def evaluate_binary(y_true, y_pred, y_score, prefix):
    return {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}_precision": precision_score(y_true, y_pred, zero_division=0),
        f"{prefix}_recall": recall_score(y_true, y_pred, zero_division=0),
        f"{prefix}_f1": f1_score(y_true, y_pred, zero_division=0),
        f"{prefix}_roc_auc": roc_auc_score(y_true, y_score),
        f"{prefix}_mcc": matthews_corrcoef(y_true, y_pred),
    }

## Fit feature selectors on training data

Mutual Information keeps the top 36 raw features. RFECV uses a common Logistic Regression ranking estimator so every downstream classifier receives the same RFECV subset. Boruta is model-specific and is therefore fitted separately with Random Forest and XGBoost. RFECV and Boruta use a reproducible stratified training-only sample for tractable runtime; final models use the complete training set.

In [ ]:
selection_size = min(SELECTION_SAMPLE_SIZE, len(X_train_combined))
selection_indices, _ = train_test_split(
    np.arange(len(X_train_combined)), train_size=selection_size,
    stratify=y_train_encoded, random_state=RANDOM_STATE,
)
X_selection = X_train_combined.iloc[selection_indices]
y_selection = y_train_encoded.iloc[selection_indices].to_numpy()
selection_preprocessor = make_selection_preprocessor()
X_selection_encoded = selection_preprocessor.fit_transform(X_selection)
encoded_feature_names = numeric_features + categorical_features
assert X_selection_encoded.shape[1] == len(encoded_feature_names)
discrete_mask = np.array([False] * len(numeric_features) + [True] * len(categorical_features))

selection_results = {
    "None": {
        "features": list(COMBINED_FEATURES),
        "details": {"method": "None"},
    }
}

mi_scores = mutual_info_classif(
    X_selection_encoded, y_selection, discrete_features=discrete_mask,
    random_state=RANDOM_STATE, n_neighbors=3,
)
mi_order = np.argsort(mi_scores)[::-1]
mi_k = min(MI_TOP_K, len(encoded_feature_names))
mi_features = [encoded_feature_names[i] for i in mi_order[:mi_k]]
selection_results["MutualInformation"] = {
    "features": mi_features,
    "details": {
        "method": "MutualInformation", "top_k": mi_k,
        "scores": {name: float(score) for name, score in zip(encoded_feature_names, mi_scores)},
    },
}

rfecv = RFECV(
    estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    step=RFECV_STEP, min_features_to_select=RFECV_MIN_FEATURES,
    cv=StratifiedKFold(RFECV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    scoring="f1", n_jobs=-1,
)
rfecv.fit(X_selection_encoded, y_selection)
rfecv_features = [
    feature for feature, keep in zip(encoded_feature_names, rfecv.support_) if keep
]
selection_results["RFECV"] = {
    "features": rfecv_features,
    "details": {
        "method": "RFECV", "estimator": "LogisticRegression",
        "folds": RFECV_FOLDS, "step": RFECV_STEP,
        "min_features": RFECV_MIN_FEATURES,
        "ranking": {name: int(rank) for name, rank in zip(encoded_feature_names, rfecv.ranking_)},
    },
}

boruta_estimators = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=7, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    ),
}
for algorithm_name, boruta_estimator in boruta_estimators.items():
    selector = BorutaPy(
        estimator=boruta_estimator, n_estimators="auto",
        max_iter=BORUTA_MAX_ITER, random_state=RANDOM_STATE, verbose=0,
    )
    selector.fit(X_selection_encoded, y_selection)
    support = selector.support_ | selector.support_weak_
    if not support.any():
        support = selector.ranking_ <= 2
    selected = [name for name, keep in zip(encoded_feature_names, support) if keep]
    selection_results[f"Boruta_{algorithm_name}"] = {
        "features": selected,
        "details": {
            "method": "Boruta", "estimator": algorithm_name,
            "max_iter": BORUTA_MAX_ITER,
            "confirmed_count": int(selector.support_.sum()),
            "tentative_count": int(selector.support_weak_.sum()),
            "ranking": {name: int(rank) for name, rank in zip(encoded_feature_names, selector.ranking_)},
        },
    }

selector_overview = pd.DataFrame([
    {
        "selector": key, "selected_count": len(value["features"]),
        "reduction_percent": 100 * (1 - len(value["features"]) / len(COMBINED_FEATURES)),
        "engineered_retained": len(set(value["features"]) & set(ENGINEERED_FEATURES)),
    }
    for key, value in selection_results.items()
]).sort_values("selected_count")
selector_overview

## Train, evaluate, and track every valid combination

`None`, Mutual Information, and RFECV are evaluated for all five classifiers. Boruta is restricted to Random Forest and XGBoost as requested. Every MLflow run stores model parameters, dataset inputs, selected and removed features, selector metadata, train/test metrics, confusion matrix, ROC curve, classification report, and the fitted preprocessing/model pipeline.

In [ ]:
models = make_models()
run_matrix = [
    (method, algorithm)
    for method in ["None", "MutualInformation", "RFECV"]
    for algorithm in models
] + [("Boruta", "RandomForest"), ("Boruta", "XGBoost")]

train_tracking_df = X_train_combined.copy()
train_tracking_df["binary_target"] = y_train
test_tracking_df = X_test_combined.copy()
test_tracking_df["binary_target"] = y_test
train_dataset = mlflow.data.from_pandas(
    train_tracking_df, source=str(data_path.resolve()),
    targets="binary_target", name="combined_train",
)
test_dataset = mlflow.data.from_pandas(
    test_tracking_df, source=str(data_path.resolve()),
    targets="binary_target", name="combined_test",
)

results = []
for feature_selection, algorithm_name in run_matrix:
    selector_key = (
        f"Boruta_{algorithm_name}" if feature_selection == "Boruta"
        else feature_selection
    )
    selector_record = selection_results[selector_key]
    selected_features = selector_record["features"]
    removed_features = [f for f in COMBINED_FEATURES if f not in selected_features]
    selected_original = [f for f in selected_features if f in ORIGINAL_FEATURES]
    selected_engineered = [f for f in selected_features if f in ENGINEERED_FEATURES]
    estimator = clone(models[algorithm_name])
    pipeline = Pipeline([
        ("preprocessor", make_model_preprocessor(selected_features)),
        ("model", estimator),
    ])
    run_name = f"Combined_{feature_selection}_{algorithm_name}"

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "data_variant": "Combined", "feature_set": "combined",
            "feature_selection": feature_selection,
            "algorithm": algorithm_name, "task": "binary_classification",
            "positive_class": "Attack",
        })
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(test_dataset, context="evaluation")
        mlflow.log_params({
            "algorithm": algorithm_name, "feature_selection": feature_selection,
            "random_state": RANDOM_STATE,
            "input_feature_count": len(COMBINED_FEATURES),
            "selected_feature_count": len(selected_features),
            "removed_feature_count": len(removed_features),
            "dimensionality_reduction_percent": 100 * (1 - len(selected_features) / len(COMBINED_FEATURES)),
            "selected_original_count": len(selected_original),
            "selected_engineered_count": len(selected_engineered),
            "selection_sample_rows": selection_size,
            "train_row_count": len(X_train_combined),
            "test_row_count": len(X_test_combined),
            "numeric_imputation": "median",
            "numeric_scaling": "StandardScaler",
            "categorical_encoding": "OneHotEncoder(handle_unknown=ignore)",
        })
        mlflow.log_params({
            f"model_{key}": value for key, value in estimator.get_params(deep=False).items()
            if value is not None
        })
        mlflow.log_dict({
            "selected_features": selected_features,
            "removed_features": removed_features,
            "selected_original_features": selected_original,
            "selected_engineered_features": selected_engineered,
        }, "feature_selection/selected_features.json")
        mlflow.log_dict(selector_record["details"], "feature_selection/selector_details.json")

        fit_started = time.perf_counter()
        pipeline.fit(X_train_combined[selected_features], y_train_encoded)
        fit_seconds = time.perf_counter() - fit_started
        inference_started = time.perf_counter()
        test_pred = pipeline.predict(X_test_combined[selected_features])
        test_score = pipeline.predict_proba(X_test_combined[selected_features])[:, 1]
        inference_seconds = time.perf_counter() - inference_started
        train_pred = pipeline.predict(X_train_combined[selected_features])
        train_score = pipeline.predict_proba(X_train_combined[selected_features])[:, 1]
        metrics = {
            **evaluate_binary(y_train_encoded, train_pred, train_score, "train"),
            **evaluate_binary(y_test_encoded, test_pred, test_score, "test"),
            "fit_seconds": fit_seconds, "test_inference_seconds": inference_seconds,
        }
        tn, fp, fn, tp = confusion_matrix(y_test_encoded, test_pred, labels=[0, 1]).ravel()
        metrics.update({
            "test_true_negatives": int(tn), "test_false_positives": int(fp),
            "test_false_negatives": int(fn), "test_true_positives": int(tp),
            "test_specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        })
        mlflow.log_metrics(metrics)
        mlflow.log_dict(classification_report(
            y_test_encoded, test_pred, labels=[0, 1],
            target_names=["Normal", "Attack"], output_dict=True, zero_division=0,
        ), "metrics/test_classification_report.json")
        mlflow.log_dict({
            "label_mapping": label_mapping,
            "original_features": list(ORIGINAL_FEATURES),
            "engineered_features": list(ENGINEERED_FEATURES),
            "library_versions": {
                "python": platform.python_version(), "mlflow": mlflow.__version__,
                "scikit_learn": sklearn.__version__,
                "ngboost": ngboost.__version__, "xgboost": xgboost.__version__,
            },
        }, "metadata/run_metadata.json")

        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(
            y_test_encoded, test_pred, display_labels=["Normal", "Attack"],
            cmap="Blues", colorbar=False, ax=ax,
        )
        ax.set_title(f"{run_name} - test confusion matrix")
        fig.tight_layout(); mlflow.log_figure(fig, "plots/test_confusion_matrix.png"); plt.close(fig)
        fig, ax = plt.subplots(figsize=(5, 4))
        RocCurveDisplay.from_predictions(y_test_encoded, test_score, name=algorithm_name, ax=ax)
        ax.set_title(f"{run_name} - test ROC curve")
        fig.tight_layout(); mlflow.log_figure(fig, "plots/test_roc_curve.png"); plt.close(fig)

        input_example = X_train_combined[selected_features].head(5).copy()
        for column in set(selected_features) & set(numeric_features):
            input_example[column] = input_example[column].astype("float64")
        mlflow.sklearn.log_model(
            sk_model=pipeline, name="model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            signature=infer_signature(input_example, pipeline.predict(input_example)),
            input_example=input_example,
        )
        results.append({
            "run_name": run_name, "run_id": run.info.run_id,
            "feature_selection": feature_selection, "algorithm": algorithm_name,
            "selected_feature_count": len(selected_features),
            "selected_engineered_count": len(selected_engineered),
            "reduction_percent": 100 * (1 - len(selected_features) / len(COMBINED_FEATURES)),
            **metrics,
        })
        print(f"Completed {run_name}: features={len(selected_features)}, test F1={metrics['test_f1']:.4f}")

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False).reset_index(drop=True)
results_df

## Research questions and evidence tables

The next cell calculates the answers directly from the experiment results and selected-feature lists, avoiding hand-written conclusions that can become stale after reruns.

In [ ]:
baseline = results_df[results_df.feature_selection == "None"].set_index("algorithm")
selected_runs = results_df[results_df.feature_selection != "None"].copy()
selected_runs["f1_delta_vs_no_selection"] = selected_runs.apply(
    lambda row: row.test_f1 - baseline.loc[row.algorithm, "test_f1"], axis=1
)
best_method_by_classifier = (
    selected_runs.sort_values("test_f1", ascending=False)
    .groupby("algorithm", as_index=False).first()[[
        "algorithm", "feature_selection", "selected_feature_count",
        "test_accuracy", "test_f1", "f1_delta_vs_no_selection"
    ]]
)

selector_instances = {
    key: value["features"] for key, value in selection_results.items() if key != "None"
}
feature_frequency = Counter(
    feature for features in selector_instances.values() for feature in features
)
feature_consistency_df = pd.DataFrame([
    {
        "feature": feature, "selected_by_count": count,
        "selected_by_percent": 100 * count / len(selector_instances),
        "feature_origin": "engineered" if feature in ENGINEERED_FEATURES else "original",
    }
    for feature, count in feature_frequency.items()
]).sort_values(["selected_by_count", "feature"], ascending=[False, True])
consistently_selected = feature_consistency_df[
    feature_consistency_df.selected_by_count == len(selector_instances)
]

engineered_retention_df = pd.DataFrame([
    {
        "selector": key,
        "selected_total": len(value["features"]),
        "engineered_selected": len(set(value["features"]) & set(ENGINEERED_FEATURES)),
        "engineered_retention_percent": 100 * len(set(value["features"]) & set(ENGINEERED_FEATURES)) / len(ENGINEERED_FEATURES),
        "selected_engineered_features": sorted(set(value["features"]) & set(ENGINEERED_FEATURES)),
    }
    for key, value in selection_results.items() if key != "None"
])

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
tracked_runs = mlflow.search_runs([experiment.experiment_id])
original_runs = tracked_runs[
    tracked_runs["tags.mlflow.runName"].fillna("").str.startswith("Original_NFS_")
].sort_values("start_time", ascending=False).drop_duplicates("tags.algorithm")
original_f1 = original_runs.set_index("tags.algorithm")["metrics.test_f1"].to_dict()
algorithm_name_map = {
    "LogisticReg": "logisticreg", "DecisionTree": "decisiontree",
    "RandomForest": "randomforest", "XGBoost": "xgboost",
    "NGBoost": "ngboost",
}
engineered_value_rows = []
for algorithm, original_name in algorithm_name_map.items():
    if original_name in original_f1:
        combined_f1 = float(baseline.loc[algorithm, "test_f1"])
        engineered_value_rows.append({
            "algorithm": algorithm, "original_test_f1": float(original_f1[original_name]),
            "combined_test_f1": combined_f1,
            "combined_minus_original_f1": combined_f1 - float(original_f1[original_name]),
        })
engineered_value_df = pd.DataFrame(engineered_value_rows)

display(Markdown("### 1. Does feature selection improve classification performance?"))
display(selected_runs[["algorithm", "feature_selection", "selected_feature_count", "test_f1", "f1_delta_vs_no_selection"]].sort_values(["algorithm", "test_f1"], ascending=[True, False]))
display(Markdown("### 2. Which feature-selection method works best for each classifier?"))
display(best_method_by_classifier)
display(Markdown("### 3. Which features are consistently selected?"))
display(consistently_selected)
display(Markdown("### 4. Are engineered features useful and retained?"))
display(engineered_value_df)
display(engineered_retention_df)
display(Markdown("### 5-6. Dimensionality reduction and the feature-count/performance trade-off"))
display(selected_runs[["algorithm", "feature_selection", "selected_feature_count", "reduction_percent", "test_accuracy", "test_f1", "f1_delta_vs_no_selection"]].sort_values("selected_feature_count"))

## Measured conclusions

### 1. Does feature selection improve classification performance?

**Sometimes, but not universally.** XGBoost improved from 99.9147% test F1 with all 72 combined features to **99.9232% with Boruta**. NGBoost improved from 98.3362% to **98.3577% with Mutual Information**. Mutual Information also raised XGBoost to 99.9190%. Random Forest with Mutual Information was effectively unchanged (a decrease of only 0.00002 percentage points) while using half the inputs. Decision Tree and Logistic Regression performed best without selection; their best selected subsets reduced F1 by approximately 0.0725 and 0.4567 percentage points, respectively.

### 2. Which selection method works best for each classifier?

| Classifier | Best selection method | Selected features | Test F1 | Change vs. no selection |
|---|---|---:|---:|---:|
| Logistic Regression | Mutual Information | 36 | 97.9054% | -0.4567 pp |
| Decision Tree | Mutual Information | 36 | 99.7954% | -0.0725 pp |
| Random Forest | Mutual Information | 36 | 99.9147% | approximately 0.0000 pp |
| XGBoost | **Boruta** | 53 | **99.9232%** | **+0.0085 pp** |
| NGBoost | Mutual Information | 36 | 98.3577% | +0.0215 pp |

For Logistic Regression and Decision Tree, `None` remains better than every feature-selection method even though Mutual Information is their best reduced alternative.

### 3. Which features are consistently selected?

The following **24 features** were selected by Mutual Information, RFECV, Random-Forest Boruta, and XGBoost Boruta:

- Engineered (11): `byte_asymmetry`, `different_host_service_connections`, `different_service_connections`, `dst_byte_fraction`, `host_error_score`, `host_scan_pressure`, `host_serror_score`, `host_service_ratio`, `same_service_rate_gap`, `short_term_scan_pressure`, `src_byte_fraction`.
- Original (13): `count`, `diffsrvrate`, `dsthostdiffsrvrate`, `dsthostsamesrcportrate`, `dsthostsamesrvrate`, `dsthostserrorrate`, `dsthostsrvcount`, `dsthostsrvdiffhostrate`, `dsthostsrvserrorrate`, `flag`, `loggedin`, `samesrvrate`, `serrorrate`.

### 4. Do engineered features improve performance and survive selection?

Compared with the matching original-feature runs, the full combined set improved Logistic Regression by **1.0962 percentage points**, Decision Tree by **0.0725 pp**, and Random Forest by **0.0085 pp** in test F1. It changed XGBoost by -0.0128 pp and NGBoost by -0.4773 pp, so engineered features are useful for some classifiers but do not guarantee an improvement for every model.

Engineered features were strongly retained: Mutual Information kept 18/31 (58.1%), RFECV 23/31 (74.2%), Random-Forest Boruta 27/31 (87.1%), and XGBoost Boruta 25/31 (80.6%). Eleven engineered features were selected by every method, providing strong evidence that the engineered variables contain reusable predictive signal.

### 5. How much dimensionality reduction is achieved without sacrificing accuracy?

Mutual Information provides the strongest compression: **72 to 36 features (50% reduction)**. It improves XGBoost and NGBoost, and leaves Random Forest test F1 effectively unchanged. RFECV reduces the input by 29.2% (72 to 51), while Boruta reduces Random Forest by 19.4% (to 58) and XGBoost by 26.4% (to 53). The clearest no-sacrifice result is therefore Mutual Information with Random Forest/XGBoost/NGBoost.

### 6. Is there a trade-off between fewer features and predictive performance?

**Yes, and it depends on the classifier.** Ensemble models tolerate aggressive reduction extremely well: XGBoost, Random Forest, and NGBoost retain or slightly improve performance with the 36-feature Mutual Information subset. Logistic Regression and Decision Tree lose measurable F1 after selection, showing that the smallest subset is not automatically optimal. When predictive differences are this small, the 50% reduction can still be preferable because it lowers input complexity, storage, and feature-computation cost.

> **Overall recommendation:** Use `Combined_Boruta_XGBoost` when maximizing test F1 is the priority. Use `Combined_MutualInformation_XGBoost` or `Combined_MutualInformation_RandomForest` when a 50% feature reduction with virtually no performance sacrifice is more valuable. All values above come from the fixed held-out test split; validate the final choice on an independent or temporal dataset before deployment.

# Final selection: Boruta with XGBoost

The best feature-selection and classifier combination is **`Combined_Boruta_XGBoost`**.

| Metric | Result |
|---|---:|
| Selected features | 53 of 72 |
| Dimensionality reduction | 26.39% |
| Test accuracy | **99.9286%** |
| Test precision | **99.9318%** |
| Test recall | **99.9147%** |
| Test F1-score | **99.9232%** |
| Test ROC-AUC | **99.9996%** |
| False positives | 8 |
| False negatives | 10 |

Boruta with XGBoost ranks first among all feature-selected combinations by test F1, accuracy, recall, and ROC-AUC. It also improves test F1 by approximately **0.0085 percentage points** over XGBoost with all 72 combined features while removing 19 features. This makes it the preferred configuration when predictive performance is the primary objective.

If operational simplicity and maximum dimensionality reduction are more important, **`Combined_MutualInformation_XGBoost`** is the alternative: it retains only 36 features (50% reduction) and achieves 99.9190% test F1, just 0.0043 percentage points below Boruta-XGBoost.

> **Decision:** Select **Boruta + XGBoost** as the best overall combination. Select **Mutual Information + XGBoost** only when halving the feature set is worth the extremely small reduction in predictive performance.